In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModel
import polars as pl

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

import polars as pl

import xgboost as xgb
from tqdm import tqdm

In [2]:
data2 = pl.read_csv('pullreq_with_code_2.csv').to_pandas()
data3 = pl.read_csv('pullreq_with_code_3.csv').to_pandas()
data4 = pl.read_csv('pullreq_with_code_4.csv').to_pandas()

d1 = data2[data2["added_code"].astype(str) != "None"]
d2 = data3[data3["added_code"].astype(str) != "None"]
d3 = data4[data4["added_code"].astype(str) != "None"]

data = pd.concat([d1, d2, d3])

In [3]:
data = data[data["added_code"].astype(str) != "None"]

In [4]:
rejected_data = data.loc[data['merged_or_not'] == 0]
rejected_data = rejected_data.loc[rejected_data['contrib_gender'].notna()]

rejected_data = rejected_data.drop(['ownername', 'reponame', 'id', 'project_id', 'github_id', 'creator_id'], axis=1)

In [5]:
le = LabelEncoder()
le.fit(rejected_data['contrib_gender'])
rejected_data['contrib_gender'] = le.transform(rejected_data['contrib_gender'])

X = rejected_data['added_code']
Y = rejected_data['contrib_gender']

In [9]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
codebert_model = AutoModel.from_pretrained("microsoft/codebert-base")

In [10]:
X_embeddings = []
for code in tqdm(X):
    inputs = tokenizer(code, return_tensors="pt", truncation=True, padding=True)
    outputs = codebert_model(**inputs)
    X_embeddings.append(outputs.last_hidden_state[:, 0, :].detach().numpy())

100%|██████████| 2078/2078 [04:27<00:00,  7.75it/s]


In [11]:
X_embeddings = np.array(X_embeddings)
X_embeddings = X_embeddings.reshape(-1, 768)

In [12]:
# split into train test val split
train_ratio = 0.70
test_ratio = 0.20
val_ratio = 0.10

X_train, X_test, y_train, y_test = train_test_split(X_embeddings, Y, test_size=1-train_ratio)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=test_ratio/(val_ratio+test_ratio))

In [13]:
X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)
X_test = np.array(X_test)
y_test = np.array(y_test)

In [14]:
X_train = X_train.reshape(-1, 768)
X_val = X_val.reshape(-1, 768)
X_test = X_test.reshape(-1, 768)

In [15]:
class Net(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size) 
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)  
    
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

In [18]:
X_train.shape

(1454, 768)

In [24]:
model = Net(input_size=768, hidden_size=500, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(100):
    for i, (codes, labels) in enumerate(zip(X_train, y_train)):
        codes = torch.tensor(codes).unsqueeze(0)
        labels = torch.tensor([labels]).long()
        
        outputs = model(codes)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    correct = 0
    total = 0
    with torch.no_grad():
        for i, (codes, labels) in enumerate(zip(X_test, y_test)):
            codes = torch.tensor(codes).unsqueeze(0)
            labels = torch.tensor([labels]).long()
            
            outputs = model(codes)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print ('Epoch [{}/{}], Train Loss: {:.4f}, Test Accuracy: {:.2f}%'.format(epoch+1, 100, loss.item(), accuracy))

Epoch [1/100], Train Loss: 0.0520, Test Accuracy: 90.14%
Epoch [2/100], Train Loss: 0.0493, Test Accuracy: 90.14%
Epoch [3/100], Train Loss: 0.0453, Test Accuracy: 90.14%
Epoch [4/100], Train Loss: 0.0462, Test Accuracy: 90.14%
Epoch [5/100], Train Loss: 0.0452, Test Accuracy: 90.14%
Epoch [6/100], Train Loss: 0.0427, Test Accuracy: 90.14%
Epoch [7/100], Train Loss: 0.0422, Test Accuracy: 90.14%
Epoch [8/100], Train Loss: 0.0400, Test Accuracy: 90.14%
Epoch [9/100], Train Loss: 0.0383, Test Accuracy: 90.14%
Epoch [10/100], Train Loss: 0.0368, Test Accuracy: 90.14%
Epoch [11/100], Train Loss: 0.0356, Test Accuracy: 90.14%
Epoch [12/100], Train Loss: 0.0346, Test Accuracy: 90.14%
Epoch [13/100], Train Loss: 0.0342, Test Accuracy: 90.14%
Epoch [14/100], Train Loss: 0.0336, Test Accuracy: 90.14%
Epoch [15/100], Train Loss: 0.0335, Test Accuracy: 90.14%
Epoch [16/100], Train Loss: 0.0327, Test Accuracy: 90.14%
Epoch [17/100], Train Loss: 0.0320, Test Accuracy: 90.14%
Epoch [18/100], Train L

In [25]:
y_true = []
y_pred = []
with torch.no_grad():
    correct = 0
    total = 0
    for i, (codes, labels) in enumerate(zip(X_val, y_val)):
        codes = torch.tensor(codes).unsqueeze(0)
        labels = torch.tensor([labels]).long()
        
        outputs = model(codes)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        y_true.extend(labels.tolist())
        y_pred.extend(predicted.tolist())

    accuracy = 100 * correct / total
    print ('Validation Accuracy: {:.2f}%'.format(accuracy))

Validation Accuracy: 88.46%


In [26]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1_score, _ = precision_recall_fscore_support(np.array(y_true), np.array(y_pred))
print(precision)
print(recall)
print(f1_score)

[0.2        0.90147783]
[0.04761905 0.97860963]
[0.07692308 0.93846154]


In [27]:
def predict_gender(code):
    inputs = tokenizer(code, return_tensors="pt", truncation=True, padding=True)
    outputs = codebert_model(**inputs)
    code_embedding = outputs.last_hidden_state[:, 0, :].detach()

    outputs = model(code_embedding)
    _, predicted = torch.max(outputs.data, 1)

    gender = 'male' if predicted.item() == 0 else 'female'

    return gender